<a href="https://colab.research.google.com/github/Muqsit069/nyc-taxi-de-project/blob/main/01_bronze_silver_gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install PySpark
!pip install pyspark -q

# Download NYC Taxi dataset
!wget -q "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet" -O nyc_taxi.parquet
print("Downloaded.")

Downloaded.


In [3]:
# Start SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYCTaxiProject") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.0.2


In [4]:
# Load the data and explore
df = spark.read.parquet("nyc_taxi.parquet")

print(f"Total rows: {df.count():,}")
print(f"Columns: {df.columns}")
df.printSchema()
df.show(5)

Total rows: 3,066,766
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- toll

In [6]:
# aggregations
from pyspark.sql.functions import (
    col, year, month, dayofmonth, hour,
    count, sum as spark_sum, avg, round as spark_round,
    when
)

df_clean = df \
    .filter(
        (col("fare_amount") > 0) &
        (col("trip_distance") > 0) &
        (col("passenger_count") > 0) &
        (col("tpep_pickup_datetime").isNotNull())
    ) \
    .withColumn("pickup_year",  year(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", month(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day",   dayofmonth(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_hour",  hour(col("tpep_pickup_datetime"))) \
    .withColumn("trip_category",
        when(col("trip_distance") < 2,  "short")
        .when(col("trip_distance") < 10, "medium")
        .otherwise("long")
    )

# Aggregation 1 — Revenue by hour of day
df_hourly = df_clean \
    .groupBy("pickup_hour") \
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("fare_amount"), 2).alias("total_fare"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance")
    ) \
    .orderBy("pickup_hour")

df_hourly.show(24)

# Aggregation 2 — Trip category breakdown
df_categories = df_clean \
    .groupBy("trip_category") \
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("tip_amount"), 2).alias("avg_tip")
    )

df_categories.show()

+-----------+-----------+----------+--------+------------+
|pickup_hour|total_trips|total_fare|avg_fare|avg_distance|
+-----------+-----------+----------+--------+------------+
|          0|      79567|1575647.62|    19.8|        4.03|
|          1|      55565| 990310.74|   17.82|        3.49|
|          2|      38731| 647680.31|   16.72|        3.22|
|          3|      25053|  444383.1|   17.74|        3.51|
|          4|      15836| 352414.03|   22.25|        4.68|
|          5|      16142| 427054.88|   26.46|        6.42|
|          6|      39945| 884784.07|   22.15|        4.78|
|          7|      79552|1505112.95|   18.92|        3.68|
|          8|     107901|1880200.71|   17.43|        3.18|
|          9|     122644|2159482.13|   17.61|        3.09|
|         10|     135285|2398188.87|   17.73|        3.14|
|         11|     145262|2528717.63|   17.41|        3.04|
|         12|     160233|2845093.22|   17.76|        3.11|
|         13|     168768| 3115934.8|   18.46|        3.2

In [7]:
# Write as partitioned Parquet
df_clean.write \
    .mode("overwrite") \
    .partitionBy("pickup_year", "pickup_month", "pickup_day") \
    .parquet("/content/nyc_taxi_silver/")

print("Written successfully.")

# Verify the partitioned structure
import os
for root, dirs, files in os.walk("/content/nyc_taxi_silver/"):
    level = root.replace("/content/nyc_taxi_silver/", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:2]:
            print(f"{indent}  {f}")

Written successfully.
/
  _SUCCESS
  ._SUCCESS.crc
pickup_year=2022/
  pickup_month=12/
    pickup_day=31/
      part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet
      .part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet.crc
  pickup_month=10/
    pickup_day=25/
      part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet
      .part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet.crc
pickup_year=2008/
  pickup_month=12/
    pickup_day=31/
      part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet
      .part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet.crc
pickup_year=2023/
  pickup_month=2/
    pickup_day=1/
      part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet
      .part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet.crc
  pickup_month=1/
    pickup_day=19/
      part-00000-0c4ce35f-cdd0-49ac-bc33-cfd4781ae262.c000.snappy.parquet
      .part-00000-0c4c